In [3]:
import os
import time
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor


# ============================================================
# 1. CONFIG
# ============================================================

# Folder of your local GitHub dataset clone
# Example structure:
# bitstamp-btcusd-minute-data/
#   data/
#     historical/btcusd_bitstamp_1min_2012-2025.csv.gz
#     updates/btcusd_bitstamp_1min_latest.csv
GIT_REPO_DIR = Path("bitstamp-btcusd-minute-data")
DATA_DIR = GIT_REPO_DIR / "data"

HISTORICAL_CSV = DATA_DIR / "historical" / "btcusd_bitstamp_1min_2012-2025.csv.gz"
RECENT_CSV = DATA_DIR / "updates" / "btcusd_bitstamp_1min_latest.csv"

# Your trained AutoGluon model folder
MODEL_PATH = "models/btc_chronos2"

# If your trained model name exists, it will use it.
# Otherwise it will automatically use AutoGluon's default/best model.
PREFERRED_MODEL_NAME = "Chronos2FineTuned"

# Must match the ID used during training
SERIES_ID = "BTC-USDT"

# The trained predictor already has prediction_length.
# This variable is only used as a fallback if predictor.prediction_length cannot be read.
FALLBACK_PREDICTION_LENGTH = 3

# Same covariates you trained with
COVARIATE_COLS = ["volume", "RSI", "MACD", "ATR", "ADX"]
# If you trained with leaner covariates, use:
# COVARIATE_COLS = ["RSI", "ADX"]

# Number of most recent rows used as model context
CONTEXT_WINDOW = 10000

# Extra rows for indicator warmup before selecting final context
INDICATOR_WARMUP_ROWS = 300

# Your data is 1-minute data
FREQ = "1min"

# If True, missing minute bars inside the context window are forward-filled
MAKE_CONTEXT_REGULAR = True

# If True, run git pull before reading CSV files
# Use True only if GIT_REPO_DIR is an actual git clone.
RUN_GIT_PULL_BEFORE_EACH_RUN = False

# Output folder
OUTPUT_DIR = Path("chronos2_github_live_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Forecast log. This stores predictions and later actual values.
FORECAST_LOG_PATH = OUTPUT_DIR / "minute_forward_forecast_log.csv"

# Latest prediction output
LATEST_FORECAST_PATH = OUTPUT_DIR / "latest_forecast.csv"

# Run mode:
#   "once" -> read data, predict next minutes once, save/log, exit
#   "loop" -> repeat forever every POLL_SECONDS
RUN_MODE = "once"

# If RUN_MODE = "loop", wait this many seconds between runs
POLL_SECONDS = 60

# If True, avoid saving duplicate predictions for same last-known timestamp
SKIP_DUPLICATE_LAST_CONTEXT = True


In [4]:

# ============================================================
# 2. SMALL UTILITIES
# ============================================================

def utc_now_naive():
    """UTC timestamp without timezone info, easier for CSV comparison."""
    return pd.Timestamp.utcnow().tz_localize(None)


def to_utc_naive_datetime(series):
    """Convert a pandas Series to timezone-naive UTC timestamps."""
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)


def clean_col_name(name):
    """Normalize column names for robust matching."""
    return "".join(ch for ch in str(name).lower().strip() if ch.isalnum())


def find_column(df, candidates, required=True):
    """
    Robustly find a column by possible names.

    Example:
        candidates = ["timestamp", "date", "time"]
    """

    normalized_to_original = {
        clean_col_name(c): c
        for c in df.columns
    }

    # Exact normalized match first
    for candidate in candidates:
        key = clean_col_name(candidate)
        if key in normalized_to_original:
            return normalized_to_original[key]

    # Containment match second
    for original in df.columns:
        original_key = clean_col_name(original)
        for candidate in candidates:
            candidate_key = clean_col_name(candidate)
            if candidate_key and candidate_key in original_key:
                return original

    if required:
        raise ValueError(
            f"Could not find required column. Candidates={candidates}. "
            f"Available columns={list(df.columns)}"
        )

    return None


def parse_timestamp_column(series):
    """
    Parses timestamp column.

    Supports:
    - UNIX seconds
    - UNIX milliseconds
    - normal datetime strings
    """

    s = series.copy()

    numeric = pd.to_numeric(s, errors="coerce")
    numeric_ratio = numeric.notna().mean()

    if numeric_ratio > 0.90:
        median_value = numeric.dropna().median()

        # milliseconds are usually > 1e12
        if median_value > 1e12:
            ts = pd.to_datetime(numeric, unit="ms", utc=True, errors="coerce")
        else:
            ts = pd.to_datetime(numeric, unit="s", utc=True, errors="coerce")
    else:
        ts = pd.to_datetime(s, utc=True, errors="coerce")

    return ts.dt.tz_convert(None)


def maybe_git_pull():
    """Optionally update the local GitHub dataset clone."""
    if not RUN_GIT_PULL_BEFORE_EACH_RUN:
        return

    if not GIT_REPO_DIR.exists():
        print(f"Git repo folder does not exist: {GIT_REPO_DIR}")
        return

    print(f"\nRunning git pull inside: {GIT_REPO_DIR}")
    try:
        result = subprocess.run(
            ["git", "-C", str(GIT_REPO_DIR), "pull", "--ff-only"],
            capture_output=True,
            text=True,
            check=False,
        )
        print(result.stdout)
        if result.stderr.strip():
            print(result.stderr)
    except Exception as e:
        print(f"git pull failed, continuing with existing local files. Error: {e}")



In [5]:

# ============================================================
# 3. LOAD AND STANDARDIZE BITSTAMP DATA
# ============================================================

def standardize_bitstamp_dataframe(raw_df):
    """
    Converts different possible Bitstamp CSV schemas into:

        id, timestamp, open, high, low, target, volume

    target = close price
    """

    df = raw_df.copy()

    timestamp_col = find_column(
        df,
        ["timestamp", "unix", "unix_timestamp", "date", "datetime", "time"],
        required=True,
    )

    open_col = find_column(df, ["open"], required=False)
    high_col = find_column(df, ["high"], required=False)
    low_col = find_column(df, ["low"], required=False)
    close_col = find_column(df, ["close", "weighted_price", "price"], required=True)

    # Prefer BTC/base volume if available
    volume_col = find_column(
        df,
        [
            "volume",
            "volume_btc",
            "volume btc",
            "volume_(btc)",
            "volumebtc",
            "base_volume",
            "amount",
        ],
        required=False,
    )

    out = pd.DataFrame()
    out["timestamp"] = parse_timestamp_column(df[timestamp_col])

    out["target"] = pd.to_numeric(df[close_col], errors="coerce")

    if open_col is not None:
        out["open"] = pd.to_numeric(df[open_col], errors="coerce")
    else:
        out["open"] = out["target"]

    if high_col is not None:
        out["high"] = pd.to_numeric(df[high_col], errors="coerce")
    else:
        out["high"] = out["target"]

    if low_col is not None:
        out["low"] = pd.to_numeric(df[low_col], errors="coerce")
    else:
        out["low"] = out["target"]

    if volume_col is not None:
        out["volume"] = pd.to_numeric(df[volume_col], errors="coerce")
    else:
        out["volume"] = 0.0

    out = out.dropna(subset=["timestamp", "target"])
    out = out.sort_values("timestamp")
    out = out.drop_duplicates(subset=["timestamp"], keep="last")
    out = out.reset_index(drop=True)

    out["id"] = SERIES_ID

    return out[["id", "timestamp", "open", "high", "low", "target", "volume"]]


def load_bitstamp_full_dataset():
    """
    Loads:

        historical/btcusd_bitstamp_1min_2012-2025.csv.gz
        updates/btcusd_bitstamp_1min_latest.csv

    then concatenates, sorts, deduplicates.
    """

    if not HISTORICAL_CSV.exists():
        raise FileNotFoundError(f"Historical CSV not found: {HISTORICAL_CSV}")

    if not RECENT_CSV.exists():
        raise FileNotFoundError(f"Recent update CSV not found: {RECENT_CSV}")

    print("\nLoading historical CSV...")
    df_hist = pd.read_csv(HISTORICAL_CSV, compression="infer")

    print("Loading recent update CSV...")
    df_recent = pd.read_csv(RECENT_CSV)

    print("Concatenating historical + recent...")
    raw_full = pd.concat([df_hist, df_recent], ignore_index=True)

    df = standardize_bitstamp_dataframe(raw_full)

    print("\n========== DATA SUMMARY ==========")
    print(f"Rows after standardization: {len(df)}")
    print(f"First timestamp:            {df['timestamp'].min()}")
    print(f"Last timestamp:             {df['timestamp'].max()}")
    print(f"Last close/target:          {df['target'].iloc[-1]:.2f}")
    print("==================================\n")

    return df


# ============================================================
# 4. TECHNICAL INDICATORS
# ============================================================

def compute_rsi(close, period=14):
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    return rsi


def compute_macd(close, fast=12, slow=26):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()

    return ema_fast - ema_slow


def compute_atr(high, low, close, period=14):
    prev_close = close.shift(1)

    tr = pd.concat(
        [
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    atr = tr.ewm(alpha=1 / period, adjust=False).mean()

    return atr


def compute_adx(high, low, close, period=14):
    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = np.where(
        (up_move > down_move) & (up_move > 0),
        up_move,
        0.0,
    )

    minus_dm = np.where(
        (down_move > up_move) & (down_move > 0),
        down_move,
        0.0,
    )

    atr = compute_atr(high, low, close, period)

    plus_di = (
        100
        * pd.Series(plus_dm, index=high.index)
        .ewm(alpha=1 / period, adjust=False)
        .mean()
        / atr.replace(0, np.nan)
    )

    minus_di = (
        100
        * pd.Series(minus_dm, index=high.index)
        .ewm(alpha=1 / period, adjust=False)
        .mean()
        / atr.replace(0, np.nan)
    )

    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)

    adx = dx.ewm(alpha=1 / period, adjust=False).mean()

    return adx


def compute_indicators(df):
    """
    Computes the same covariates used in your original training strategy.
    """

    df = df.copy()

    if "RSI" in COVARIATE_COLS:
        df["RSI"] = compute_rsi(df["target"])

    if "MACD" in COVARIATE_COLS:
        df["MACD"] = compute_macd(df["target"])

    if "ATR" in COVARIATE_COLS:
        df["ATR"] = compute_atr(df["high"], df["low"], df["target"])

    if "ADX" in COVARIATE_COLS:
        df["ADX"] = compute_adx(df["high"], df["low"], df["target"])

    df = df.replace([np.inf, -np.inf], np.nan)

    return df


# ============================================================
# 5. PREPARE MODEL CONTEXT
# ============================================================

def make_regular_1min_context(df):
    """
    Makes the latest context regular at 1-minute frequency.

    Missing target/open/high/low are forward-filled.
    Missing volume is set to 0.
    """

    df = df.copy()
    df = df.sort_values("timestamp")
    df = df.drop_duplicates("timestamp", keep="last")

    df = df.set_index("timestamp")

    full_index = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq=FREQ,
    )

    df = df.reindex(full_index)
    df.index.name = "timestamp"

    df["target"] = df["target"].ffill()
    df["open"] = df["open"].fillna(df["target"])
    df["high"] = df["high"].fillna(df["target"])
    df["low"] = df["low"].fillna(df["target"])
    df["volume"] = df["volume"].fillna(0.0)
    df["id"] = SERIES_ID

    df = df.dropna(subset=["target"])

    return df.reset_index()


def prepare_context_for_prediction(df_full):
    """
    Selects latest rows, regularizes them, computes indicators,
    and returns the final TimeSeriesDataFrame-compatible context.
    """

    needed_raw_rows = CONTEXT_WINDOW + INDICATOR_WARMUP_ROWS

    df_context = df_full.sort_values("timestamp").tail(needed_raw_rows).copy()

    if MAKE_CONTEXT_REGULAR:
        df_context = make_regular_1min_context(df_context)

    df_context = compute_indicators(df_context)

    required_cols = ["id", "timestamp", "target"] + COVARIATE_COLS

    missing_cols = [c for c in required_cols if c not in df_context.columns]
    if missing_cols:
        raise ValueError(f"Missing columns after indicator computation: {missing_cols}")

    df_context = df_context[required_cols].dropna().reset_index(drop=True)

    df_context = df_context.tail(CONTEXT_WINDOW).reset_index(drop=True)

    if len(df_context) == 0:
        raise RuntimeError("Context is empty after computing indicators and dropping NaNs.")

    print("\n========== CONTEXT SUMMARY ==========")
    print(f"Context rows:       {len(df_context)}")
    print(f"Context first time: {df_context['timestamp'].min()}")
    print(f"Context last time:  {df_context['timestamp'].max()}")
    print(f"Last known price:   {df_context['target'].iloc[-1]:.2f}")
    print("=====================================\n")

    return df_context


In [6]:


# ============================================================
# 6. LOAD MODEL AND FORECAST
# ============================================================

def load_saved_predictor():
    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(f"MODEL_PATH does not exist: {MODEL_PATH}")

    predictor = TimeSeriesPredictor.load(MODEL_PATH)

    print("\n========== LOADED PREDICTOR ==========")
    print(f"Model path:          {MODEL_PATH}")

    prediction_length = getattr(
        predictor,
        "prediction_length",
        FALLBACK_PREDICTION_LENGTH,
    )

    print(f"Prediction length:   {prediction_length}")

    try:
        model_names = predictor.model_names()
        print(f"Available models:    {model_names}")
    except Exception:
        model_names = []
        print("Available models:    could not read model_names()")

    print("======================================\n")

    return predictor


def choose_model_name(predictor):
    """
    Uses PREFERRED_MODEL_NAME if available.
    Otherwise returns None, which means AutoGluon's default/best model.
    """

    if PREFERRED_MODEL_NAME is None:
        return None

    try:
        names = predictor.model_names()
    except Exception:
        names = []

    if PREFERRED_MODEL_NAME in names:
        print(f"Using preferred model: {PREFERRED_MODEL_NAME}")
        return PREFERRED_MODEL_NAME

    print(
        f"Preferred model '{PREFERRED_MODEL_NAME}' was not found. "
        f"Using AutoGluon's default/best model."
    )
    return None


def select_prediction_column(forecast_df):
    """
    Selects forecast column.

    Chronos / AutoGluon often provides:
        mean
        0.1, 0.2, 0.5, 0.8, 0.9

    We prefer mean, then 0.5.
    """

    for col in ["mean", "0.5", "median"]:
        if col in forecast_df.columns:
            return col

    ignore_cols = {"item_id", "id", "timestamp"}

    numeric_cols = [
        c
        for c in forecast_df.columns
        if c not in ignore_cols and pd.api.types.is_numeric_dtype(forecast_df[c])
    ]

    if not numeric_cols:
        raise ValueError(f"No numeric prediction columns found in forecast: {forecast_df.columns}")

    return numeric_cols[0]


def predict_next_minutes(predictor, model_name, df_context):
    """
    Runs Chronos-2 prediction and returns one row per future minute.
    """

    input_cols = ["id", "timestamp", "target"] + COVARIATE_COLS

    context_ts = TimeSeriesDataFrame.from_data_frame(
        df_context[input_cols],
        id_column="id",
        timestamp_column="timestamp",
    )

    if model_name is None:
        forecast = predictor.predict(context_ts)
    else:
        forecast = predictor.predict(context_ts, model=model_name)

    forecast_df = forecast.reset_index()

    # AutoGluon normally uses "item_id" after reset_index()
    if "item_id" in forecast_df.columns:
        item_col = "item_id"
    elif "id" in forecast_df.columns:
        item_col = "id"
    else:
        item_col = None

    if item_col is not None:
        forecast_df = forecast_df[forecast_df[item_col] == SERIES_ID].copy()

    if "timestamp" not in forecast_df.columns:
        raise ValueError(
            f"Could not find timestamp column in forecast output. "
            f"Columns={forecast_df.columns}"
        )

    pred_col = select_prediction_column(forecast_df)

    forecast_df = forecast_df.sort_values("timestamp").reset_index(drop=True)

    last_known_timestamp = df_context["timestamp"].iloc[-1]
    last_known_price = float(df_context["target"].iloc[-1])
    generated_at = utc_now_naive()

    rows = []

    for i, row in forecast_df.iterrows():
        target_timestamp = row["timestamp"]
        predicted_price = float(row[pred_col])

        predicted_change = predicted_price - last_known_price
        predicted_return_pct = 100.0 * predicted_change / last_known_price

        if predicted_change > 0:
            direction = "UP"
        elif predicted_change < 0:
            direction = "DOWN"
        else:
            direction = "FLAT"

        step_ahead = i + 1
        step_ahead_minutes = (
            pd.Timestamp(target_timestamp) - pd.Timestamp(last_known_timestamp)
        ) / pd.Timedelta(minutes=1)

        rows.append(
            {
                "forecast_generated_at": generated_at,
                "model_name": model_name if model_name is not None else "AutoGluonDefaultBest",
                "series_id": SERIES_ID,
                "last_known_timestamp": last_known_timestamp,
                "last_known_price": last_known_price,
                "target_timestamp": target_timestamp,
                "step_ahead": int(step_ahead),
                "step_ahead_minutes": float(step_ahead_minutes),
                "predicted_price": predicted_price,
                "predicted_change": predicted_change,
                "predicted_return_pct": predicted_return_pct,
                "predicted_direction": direction,
                "prediction_column_used": pred_col,

                # Filled later when real data becomes available
                "actual_price": np.nan,
                "error": np.nan,
                "absolute_error": np.nan,
                "absolute_pct_error": np.nan,
                "actual_direction": None,
                "direction_correct": np.nan,
                "scored_at": pd.NaT,
            }
        )

    prediction_df = pd.DataFrame(rows)

    return prediction_df


# ============================================================
# 7. SCORE PREVIOUS FORECASTS WHEN FUTURE DATA EXISTS
# ============================================================

def load_existing_forecast_log():
    if not FORECAST_LOG_PATH.exists():
        return pd.DataFrame()

    log_df = pd.read_csv(FORECAST_LOG_PATH)

    datetime_cols = [
        "forecast_generated_at",
        "last_known_timestamp",
        "target_timestamp",
        "scored_at",
    ]

    for col in datetime_cols:
        if col in log_df.columns:
            log_df[col] = pd.to_datetime(log_df[col], errors="coerce")

    return log_df


def score_forecast_log(log_df, df_full):
    """
    For predictions whose target_timestamp now exists in the dataset,
    fill actual_price and error metrics.
    """

    if log_df.empty:
        return log_df

    df_actual = (
        df_full[["timestamp", "target"]]
        .drop_duplicates("timestamp", keep="last")
        .rename(columns={"timestamp": "target_timestamp", "target": "actual_price_new"})
        .copy()
    )

    log_df = log_df.copy()

    log_df["target_timestamp"] = to_utc_naive_datetime(log_df["target_timestamp"])
    log_df["last_known_timestamp"] = to_utc_naive_datetime(log_df["last_known_timestamp"])
    log_df["forecast_generated_at"] = to_utc_naive_datetime(log_df["forecast_generated_at"])

    df_actual["target_timestamp"] = to_utc_naive_datetime(df_actual["target_timestamp"])

    merged = log_df.merge(
        df_actual,
        on="target_timestamp",
        how="left",
    )

    if "actual_price" not in merged.columns:
        merged["actual_price"] = np.nan

    needs_score = merged["actual_price"].isna() & merged["actual_price_new"].notna()

    merged.loc[needs_score, "actual_price"] = merged.loc[needs_score, "actual_price_new"]

    has_actual = merged["actual_price"].notna()

    merged.loc[has_actual, "error"] = (
        merged.loc[has_actual, "predicted_price"]
        - merged.loc[has_actual, "actual_price"]
    )

    merged.loc[has_actual, "absolute_error"] = merged.loc[has_actual, "error"].abs()

    merged.loc[has_actual, "absolute_pct_error"] = (
        100.0
        * merged.loc[has_actual, "absolute_error"]
        / merged.loc[has_actual, "actual_price"].replace(0, np.nan)
    )

    actual_delta = merged.loc[has_actual, "actual_price"] - merged.loc[has_actual, "last_known_price"]

    merged.loc[has_actual, "actual_direction"] = np.where(
        actual_delta > 0,
        "UP",
        np.where(actual_delta < 0, "DOWN", "FLAT"),
    )

    merged.loc[has_actual, "direction_correct"] = (
        merged.loc[has_actual, "actual_direction"]
        == merged.loc[has_actual, "predicted_direction"]
    )

    now = utc_now_naive()

    if "scored_at" not in merged.columns:
        merged["scored_at"] = pd.NaT

    merged.loc[needs_score, "scored_at"] = now

    merged = merged.drop(columns=["actual_price_new"])

    return merged


def append_new_predictions_and_score(new_predictions, df_full):
    """
    Appends new predictions to the CSV log, scores all possible previous rows,
    removes duplicate forecasts if configured, and saves.
    """

    old_log = load_existing_forecast_log()

    if old_log.empty:
        combined = new_predictions.copy()
    else:
        combined = pd.concat([old_log, new_predictions], ignore_index=True)

    if SKIP_DUPLICATE_LAST_CONTEXT:
        subset_cols = [
            "model_name",
            "series_id",
            "last_known_timestamp",
            "target_timestamp",
            "step_ahead",
        ]

        existing_subset_cols = [c for c in subset_cols if c in combined.columns]

        combined = combined.drop_duplicates(
            subset=existing_subset_cols,
            keep="last",
        ).reset_index(drop=True)

    combined = score_forecast_log(combined, df_full)

    combined.to_csv(FORECAST_LOG_PATH, index=False)

    return combined


def print_score_summary(log_df):
    if log_df.empty or "actual_price" not in log_df.columns:
        print("No forecast log yet.")
        return

    scored = log_df[log_df["actual_price"].notna()].copy()

    if scored.empty:
        print("\nNo previous predictions have actual future prices yet.")
        return

    print("\n========== SCORED FORECAST SUMMARY ==========")

    overall_mae = scored["absolute_error"].mean()
    overall_rmse = np.sqrt((scored["error"] ** 2).mean())
    overall_mape = scored["absolute_pct_error"].mean()
    direction_accuracy = scored["direction_correct"].mean()

    print(f"Scored rows:          {len(scored)}")
    print(f"Overall MAE:          {overall_mae:.4f}")
    print(f"Overall RMSE:         {overall_rmse:.4f}")
    print(f"Overall MAPE:         {overall_mape:.4f}%")
    print(f"Direction accuracy:   {direction_accuracy:.2%}")

    print("\nBy forecast step:")

    by_step = (
        scored.groupby("step_ahead")
        .agg(
            rows=("absolute_error", "size"),
            mae=("absolute_error", "mean"),
            rmse=("error", lambda x: np.sqrt(np.mean(np.square(x)))),
            mape=("absolute_pct_error", "mean"),
            direction_accuracy=("direction_correct", "mean"),
        )
        .reset_index()
    )

    print(by_step.to_string(index=False))

    print("=============================================\n")


# ============================================================
# 8. ONE COMPLETE RUN
# ============================================================

def run_one_forecast_cycle(predictor, model_name):
    """
    One complete live-forward-test cycle:

    1. Optionally git pull
    2. Load latest GitHub CSV data
    3. Prepare latest context
    4. Predict next minutes
    5. Append predictions to log
    6. Score old predictions if future actual data exists
    """

    maybe_git_pull()

    df_full = load_bitstamp_full_dataset()

    df_context = prepare_context_for_prediction(df_full)

    new_predictions = predict_next_minutes(
        predictor=predictor,
        model_name=model_name,
        df_context=df_context,
    )

    new_predictions.to_csv(LATEST_FORECAST_PATH, index=False)

    combined_log = append_new_predictions_and_score(
        new_predictions=new_predictions,
        df_full=df_full,
    )

    print("\n========== NEW MINUTE-BY-MINUTE FORECAST ==========")

    display_cols = [
        "forecast_generated_at",
        "last_known_timestamp",
        "last_known_price",
        "target_timestamp",
        "step_ahead",
        "step_ahead_minutes",
        "predicted_price",
        "predicted_change",
        "predicted_return_pct",
        "predicted_direction",
    ]

    print(new_predictions[display_cols].to_string(index=False))

    print("===================================================\n")

    print_score_summary(combined_log)

    print(f"Saved latest forecast to: {LATEST_FORECAST_PATH}")
    print(f"Saved full forecast log to: {FORECAST_LOG_PATH}")

    return new_predictions, combined_log



In [7]:
def main():
    predictor = load_saved_predictor()
    model_name = choose_model_name(predictor)

    if RUN_MODE == "once":
        run_one_forecast_cycle(predictor, model_name)

    elif RUN_MODE == "loop":
        print(f"\nStarting loop mode. Polling every {POLL_SECONDS} seconds. Ctrl+C to stop.")

        while True:
            try:
                run_one_forecast_cycle(predictor, model_name)
            except Exception as e:
                print(f"\nForecast cycle failed. Will retry next cycle. Error:\n{e}")

            time.sleep(POLL_SECONDS)

    else:
        raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

In [8]:
if __name__ == "__main__":
    main()


========== LOADED PREDICTOR ==========
Model path:          models/btc_chronos2
Prediction length:   3
Available models:    ['Chronos2ZeroShot', 'Chronos2FineTuned']

Using preferred model: Chronos2FineTuned

Loading historical CSV...
Loading recent update CSV...
Concatenating historical + recent...

========== DATA SUMMARY ==========
Rows after standardization: 7651634
First timestamp:            2012-01-01 10:01:00
Last timestamp:             2026-07-20 01:14:00
Last close/target:          64752.42


========== CONTEXT SUMMARY ==========
Context rows:       10000
Context first time: 2026-07-13 02:35:00
Context last time:  2026-07-20 01:14:00
Last known price:   64752.42


========== NEW MINUTE-BY-MINUTE FORECAST ==========
     forecast_generated_at last_known_timestamp  last_known_price    target_timestamp  step_ahead  step_ahead_minutes  predicted_price  predicted_change  predicted_return_pct predicted_direction
2026-07-20 09:14:34.416287  2026-07-20 01:14:00          64752.42 202

/tmp/ipykernel_624264/2380216402.py:270: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[has_actual, "direction_correct"] = (
